# OpenGeoStreams walkthrough

Demonstrates every public function in the `opengeostreams` library on the
bundled N-choe dataset.

**Every cell runs on its own.** Each one starts with
`from walkthrough_setup import *`, which gives it `data` (the full
`consolidated_dataset.csv`), `ogs`, `pd`, `CSV`, `OUTPUT` and `PARAMETER`.
The dataset is loaded only the first time, so run cells in any order.
Shared settings live in [walkthrough_setup.py](walkthrough_setup.py).

Optional sections (PNG export, elevations, dashboard) have a switch at the top
of their cell.

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
print("opengeostreams", ogs.__version__)
print("CSV:", CSV)
print("loaded rows:", len(data))

## 1. Load a CSV
`load_csv(path, rows=400, merge_stations=True)`. Pass `rows="all"` for the whole file.

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
first_400 = ogs.load_csv(CSV)  # first 400 records (ogs.MAX_PAGE_SIZE)
print(len(first_400), "loaded of", first_400.total_rows)
print("loaded_rows:", first_400.loaded_rows,
      "| remaining_rows:", first_400.remaining_rows,
      "| has_more:", first_400.has_more)

first_1000 = ogs.load_csv(CSV, rows=1000)
print(len(first_1000), "loaded of", first_1000.total_rows)

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
data.head(10)

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
# The processed measurements as a regular pandas DataFrame
data.data.sample(5, random_state=0)

## 2. Summary and statistics

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
summary = data.summary
{k: v for k, v in summary.items() if k not in {"missingByColumn", "mergedLocations", "coordinateConflicts"}}

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
pd.Series(data.summary['missingByColumn'], name='empty cells')

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
pd.DataFrame(data.summary['mergedLocations'])

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
data.describe()

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
data.describe(parameter=PARAMETER)

## 3. Data-quality checks
`coordinate_conflicts()` lists stations whose rows give coordinates more than `min_spread_m` apart.

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
conflicts = data.coordinate_conflicts(min_spread_m=500)
print(len(conflicts), "station(s) with conflicting coordinates")
conflicts

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
# Compare with label merging turned off
raw = ogs.load_csv(CSV, rows="all", merge_stations=False)
print("stations with merging:", data.data["Location"].nunique(),
      "| without:", raw.data["Location"].nunique())

## 4. Build datasets from DataFrames and combine them
`from_dataframe()` and `concat()` return new `RiverDataset` objects.

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
frame = data.data.copy()
first_half = ogs.from_dataframe(frame.iloc[: len(frame) // 2], name="first half")
second_half = ogs.from_dataframe(frame.iloc[len(frame) // 2 :], name="second half")
print(len(first_half), len(second_half), first_half.path)

combined = ogs.concat([first_half, second_half], name="recombined")
print("recombined:", len(combined), "rows")

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
# DataFrames can be passed directly, and `station=` gives every row one station name
frame = data.data
one_site = ogs.concat([frame[frame["Location"].str.contains("Sector 83")]], station="Sector 83")
one_site.data["Location"].unique()

## 5. Export to CSV
`to_csv()` returns CSV text when no path is given.

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
print(data.to_csv()[:400])
print("written:", data.to_csv(OUTPUT / "notebook_export.csv"))

## 6. Drain/river network
`channel_network()` matches stations to drains/rivers from the bundled India-WRIS
Punjab channels and orders them upstream → downstream. Direction uses Open-Meteo
elevations when online; pass `use_elevation=False` if you are offline.

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
network = data.channel_network(max_distance_km=2.0)
print("channels:", len(network["channels"]), "| unassigned stations:", len(network["unassigned"]))

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
# One row per station with its drain/river and upstream order
data.assign_channels()

## 7. Interpolation along the rivers
Estimates follow the drains and rivers in the direction of flow: a point uses only stations
that share water with it, weighted by that water share and by distance along the channel.
By default the distance limit is chosen by leave-one-out (`max_distance_km="auto"`);
pass a number to fix it. `advanced=True` also tries stream-network kriging when there are
at least 50 stations.

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
_, interp = data.interpolate()  # or data.interpolate(max_distance_km=5)
interp

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
# How the distance limit was chosen: lower relative error is better
selection = data._interpolation["method"]["distanceSelection"]
print("chosen limit (km):", selection["chosenKm"], "| beats the station average:", selection["beatsBaseline"])
pd.DataFrame(selection["candidates"])

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
# Self-check per parameter: skill above 0 means the map beats the average of the other stations
pd.DataFrame([{"parameter": s["parameter"], "method": s["method"], **{k: s["validation"][k] for k in
               ("checkedCount", "rmse", "baselineRmse", "skill")}}
              for s in data._interpolation["surfaces"]["allYears"] if s.get("validation")])

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
# The river network used: source, outlets and how each station was placed on it
network_info = data._interpolation["network"]
print({k: network_info.get(k) for k in ("source", "junctionWeights", "splits", "treeShaped")})
pd.DataFrame(network_info.get("stations", []))

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
# All years (runs interpolate() automatically if needed)
data.plot(parameter=PARAMETER)

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
# One year's surface; pick from the years in the data
years = sorted(data.data["Date"].str.extract(r"(\d{4})")[0].dropna().astype(int).unique().tolist())
print("years:", years)
data.plot(parameter=PARAMETER, year=years[-1])

## 8. Plots
All charts are interactive Plotly figures; hover for details.

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
data.plot_trend(parameter=PARAMETER)

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
data.plot_trend(parameter=PARAMETER, location=data.data['Location'].iloc[0])

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
data.plot_distribution(parameter=PARAMETER, grouping='year')

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
data.plot_distribution(parameter=PARAMETER, grouping='month')

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
data.plot_stream(parameter=PARAMETER)

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
data.plot_stream(parameter=PARAMETER, group_by_channel=True)

## 9. Map
`plot_map()` returns a Folium map, which renders inline. `open_browser=False` stops it opening a browser tab.

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
station_map = data.plot_map(parameter=PARAMETER, save_path=OUTPUT / "station_map.html", open_browser=False)
print("saved:", station_map.path)
station_map

## 10. Save figures
Any plot method accepts `save_path`. `.html` always works; `.png` needs the image extras.

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
data.plot_trend(parameter=PARAMETER, save_path=OUTPUT / "trend.html")
print("saved", OUTPUT / "trend.html")

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
SAVE_PNG = False  # set True: needs `pip install "opengeostreams[images]"` and Chrome

if SAVE_PNG:
    for name in ("plot", "plot_trend", "plot_distribution", "plot_stream", "plot_map"):
        options = {"open_browser": False} if name == "plot_map" else {}
        getattr(data, name)(parameter=PARAMETER, save_path=OUTPUT / f"{name}.png", **options)
    print("PNGs saved in", OUTPUT)
else:
    print("Skipped: set SAVE_PNG = True to export PNGs")

## 11. Lower-level helpers
Useful building blocks from `opengeostreams.network` and `opengeostreams.pipeline`.

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
from opengeostreams import network as ogs_network, pipeline as ogs_pipeline

print(ogs_network.parse_water_body("NCP06 - N-choe before confluence with Ghaggar"))
print(ogs_network.canonical_name("N Choe"))
print(ogs_network.name_similarity("N-choe", "N Choe"))
print(ogs_pipeline.normalize_parameter("PH"))
print(ogs_pipeline.parse_date_bounds("[01-01-2018, 31-01-2018]"))
print(len(ogs_network.load_aliases()), "water-body aliases")
ogs_network.station_table(data.data).head()

## 12. Station elevations (optional)
Needs `opengeostreams[earthengine]` and an Earth Engine project in `OPENGEOSTREAMS_EE_PROJECT`.

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
RUN_ELEVATION = False  # set True once Earth Engine is configured

if RUN_ELEVATION:
    display(data.fetch_elevations())
else:
    print("Skipped: set RUN_ELEVATION = True to fetch elevations")

## 13. Dashboard (optional)
Starts the dashboard in a background thread. Run the next cell to stop it.

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
RUN_DASHBOARD = False  # set True to start the dashboard

if RUN_DASHBOARD:
    server = data.dashboard(port=4173, open_browser=True)
    print("Dashboard running at", server.url)
else:
    print("Skipped: set RUN_DASHBOARD = True to start the dashboard")

In [ ]:
from walkthrough_setup import *  # loads `data` once; makes this cell runnable on its own
# Stop the dashboard started above
if "server" in globals():
    server.shutdown()
    server.server_close()
    del server
    print("Dashboard stopped")
else:
    print("No dashboard is running")